<img src="https://raw.githubusercontent.com/MeMoModelling/proteinSeqMapping/main/bii_horizontal_logo_smalle472014210204e8886d5cf09c653e7e5.png" width="160" align="right"/>

# Protein Sequence Mapping

Maps metabolic model protein sequences to a genome annotation via BLASTp, outputting an ID mapping CSV for use with the `identifiers_mapping` tool.

<br>

### Instructions:
1. Install BLAST using one of the options in the **Install packages** section.
2. In the **Inputs** section, replace the file paths with the paths to your own FASTA files.
3. Run all cells top to bottom (**Kernel → Restart and Run All**).

In [ ]:
#@title Step 1 — Install BLAST { display-mode: "form" }
from IPython.display import display, HTML
display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600;700&display=swap');
body, p, li, span, div { font-family: 'DM Sans', sans-serif !important; color: #0d0d0d; }
.nb-step { background: white; border: 1.5px solid #e8e8e8; border-radius: 8px; padding: 1.2rem 1.5rem; margin-bottom: 1rem; }
.nb-step h3 { font-size: 1rem; font-weight: 700; text-transform: uppercase; letter-spacing: 0.5px; color: #006d5b; margin: 0 0 0.4rem 0; }
.nb-step p { margin: 0; color: #333; font-size: 0.9rem; line-height: 1.5; }
</style>
"""))
display(HTML("<div class='nb-step'><h3>Step 1 &mdash; Install BLAST</h3><p>Installing required tools. This runs automatically and takes about 1 minute.</p></div>"))
import shutil, subprocess
if not shutil.which('blastp'):
    print('Installing BLAST...')
    subprocess.run(['apt-get', 'install', '-y', '-q', 'ncbi-blast+'], check=True)
    print('\u2705 BLAST installed.')
else:
    print('\u2705 BLAST already installed.')
subprocess.run(['pip', 'install', '-q', 'pandas'], capture_output=True)

In [ ]:
#@title Step 2 — Upload FASTA files { display-mode: "form" }
from IPython.display import display, HTML
from google.colab import files
import os

display(HTML("<div class='nb-step'><h3>Step 2 &mdash; Upload your FASTA files</h3><p>Upload your <strong>query FASTA</strong> (model protein sequences) then your <strong>database FASTA</strong> (NCBI protein sequences for the same species).</p></div>"))

if 'query_filename' in globals() and os.path.exists(query_filename):
    print(f'\u2705 Using demo files: {query_filename} / {db_filename}')
else:
    print('\ud83d\udcc2  Upload QUERY FASTA \u2014 protein sequences from your metabolic model')
    uploaded_query = files.upload()
    query_filename_raw = list(uploaded_query.keys())[0]
    query_filename = query_filename_raw.replace(' ', '_')
    if query_filename != query_filename_raw:
        os.rename(query_filename_raw, query_filename)
    print(f'\u2705 Query: {query_filename}')

    print('\n\ud83d\udcc2  Upload DATABASE FASTA \u2014 protein sequences downloaded from NCBI')
    uploaded_db = files.upload()
    db_filename_raw = list(uploaded_db.keys())[0]
    db_filename = db_filename_raw.replace(' ', '_')
    if db_filename != db_filename_raw:
        os.rename(db_filename_raw, db_filename)
    print(f'\u2705 Database: {db_filename}')

    species_name = query_filename.split('_')[0]

display(HTML(f"<div style='margin-top:0.8rem;padding:0.6rem 1rem;background:#f0fff8;border-left:3px solid #006d5b;border-radius:0 6px 6px 0;font-size:0.9rem;'>\ud83d\udd2c Species detected: <strong>{species_name}</strong></div>"))

In [ ]:
#@title Step 3 — Run BLAST { display-mode: "form" }
from IPython.display import display, HTML
import subprocess, pandas as pd, os

display(HTML("<div class='nb-step'><h3>Step 3 &mdash; Run BLAST &amp; generate mapping CSV</h3><p>Running BLASTp and filtering results. This may take several minutes.</p></div>"))

db_name    = f'{species_name}_db'
blast_out  = f'{species_name}_blast.txt'
output_csv = f'{species_name}_protein_id_mapping.csv'

print('\u2699\ufe0f  Building BLAST database...')
subprocess.run(['makeblastdb', '-in', db_filename, '-dbtype', 'prot', '-out', db_name],
               check=True, capture_output=True)

print('\u2699\ufe0f  Running BLASTp...')
subprocess.run([
    'blastp', '-query', query_filename, '-db', db_name, '-out', blast_out,
    '-outfmt', '6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore qcovs',
    '-num_threads', '2'
], check=True, capture_output=True)

cols = ['qseqid','sseqid','pident','length','mismatch','gapopen','qstart','qend','sstart','send','evalue','bitscore','qcovs']
blast_df = pd.read_csv(blast_out, sep='\t', header=None, names=cols)

def seq_lengths(fasta):
    lengths, cur_id, cur_len = {}, None, 0
    with open(fasta) as f:
        for line in f:
            if line.startswith('>'):
                if cur_id: lengths[cur_id] = cur_len
                cur_id, cur_len = line[1:].strip().split()[0], 0
            else:
                cur_len += len(line.strip())
    if cur_id: lengths[cur_id] = cur_len
    return lengths

blast_df['qlen'] = blast_df['qseqid'].map(seq_lengths(query_filename))
blast_df['slen'] = blast_df['sseqid'].map(seq_lengths(db_filename))
blast_df['qcov'] = (blast_df['qend'] - blast_df['qstart'] + 1) / blast_df['qlen']
blast_df['scov'] = blast_df['length'] / blast_df['slen']

filtered = blast_df[
    (blast_df['pident']   >= 95)   &
    (blast_df['gapopen']  == 0)    &
    (blast_df['mismatch'] <= 1)    &
    (blast_df['qcov']     >= 0.85) &
    (blast_df['scov']     >= 0.85)
]

best   = filtered.sort_values('evalue').groupby('qseqid').first().reset_index()
result = best[['qseqid','sseqid']].rename(columns={'qseqid':'identifier_query','sseqid':'identifier_db'})
result.to_csv(output_csv, index=False)

display(HTML(f"""
<div style='margin-top:1rem;padding:1rem 1.5rem;background:#f0fff8;border:1.5px solid #006d5b;border-radius:8px;'>
  <div style='font-size:1rem;font-weight:700;color:#006d5b;margin-bottom:0.4rem;'>\u2705 Done!</div>
  <div style='font-size:0.9rem;color:#333;'>{len(result)} matches found out of {blast_df['qseqid'].nunique()} sequences.</div>
  <div style='font-size:0.9rem;color:#333;margin-top:0.3rem;'>Output saved as <strong>{output_csv}</strong></div>
  <div style='font-size:0.9rem;color:#555;margin-top:0.6rem;'>&#128193; Find the file in the <strong>Files panel on the left</strong> &rarr; right-click &rarr; <strong>Download</strong></div>
</div>
"""))